# TimesFM Diagnostics\nStep-by-step inspection of why `predict_quantiles` returns `None`.

## 1. Load model

In [3]:
import os, importlib.util
import numpy as np
import torch
import timesfm

spec = importlib.util.spec_from_file_location("local_secrets", "./secrets.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
os.environ["HF_TOKEN"] = mod.HF_token

torch.set_float32_matmul_precision("high")

model = timesfm.TimesFM_1p3_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch", torch_compile=False
)
model.compile(
    timesfm.ForecastConfig(
        max_context=512,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)
print("Model loaded OK")

AttributeError: module 'timesfm' has no attribute 'TimesFM_1p3_200M_torch'

## 2. Raw `forecast` output — inspect types and shapes

In [ ]:
rng = np.random.default_rng(42)
history = rng.standard_normal(400).astype(np.float32)
horizon = 1

raw = model.forecast(horizon=horizon, inputs=[history])

print("type(raw)      :", type(raw))
print("len(raw)       :", len(raw))
for i, item in enumerate(raw):
    print(f"  raw[{i}] type  : {type(item)}")
    if hasattr(item, 'shape'):
        print(f"  raw[{i}] shape : {item.shape}")
    elif item is None:
        print(f"  raw[{i}]       : None")
    else:
        try:
            arr = np.asarray(item)
            print(f"  raw[{i}] asarray shape: {arr.shape}, dtype: {arr.dtype}")
        except Exception as e:
            print(f"  raw[{i}] asarray failed: {e}")

## 3. Walk through `predict_quantiles` logic step-by-step

In [ ]:
point_forecast, qfan = raw

print("=== point_forecast ===")
print("  type :", type(point_forecast))
print("  shape:", np.asarray(point_forecast).shape)
print("  [0, :horizon]:", np.asarray(point_forecast)[0, :horizon])

print()
print("=== qfan ===")
print("  type :", type(qfan))

if qfan is None:
    print("  >>> qfan is None — quantile head not returning anything <<<")
else:
    qarr = np.asarray(qfan)
    print("  shape:", qarr.shape)
    print("  dtype:", qarr.dtype)
    print("  ndim :", qarr.ndim)
    if qarr.ndim == 3:
        slab = qarr[0, :horizon, :]
        k = slab.shape[-1]
        print("  slab shape (after [0, :h, :]):", slab.shape)
        print("  K (last dim):", k)
        if k >= 10:
            qvals = slab[:, 1:10].T
            print("  Branch: k>=10 → qvals.shape:", qvals.shape, "  expected (9, horizon):", (9, horizon))
        elif k == 9:
            qvals = slab[:, :9].T
            print("  Branch: k==9 → qvals.shape:", qvals.shape, "  expected (9, horizon):", (9, horizon))
        else:
            print(f"  >>> k={k} — neither >=10 nor ==9, returns None <<<")
    else:
        print("  >>> ndim != 3, returns None <<<")

## 4. Check if calling `forecast` twice in a row (as the runner does) changes anything

In [ ]:
# The runner calls fit_predict (→ forecast) then predict_quantiles (→ forecast again).
# Check whether the second call returns the same structure.

raw2 = model.forecast(horizon=horizon, inputs=[history])
pf2, qf2 = raw2
print("Second call qfan type :", type(qf2))
if qf2 is not None:
    print("Second call qfan shape:", np.asarray(qf2).shape)
else:
    print(">>> qfan is None on second call too")

## 5. End-to-end through the actual `TimesFMForecaster`

In [ ]:
from benchmark.forecasters.timesfm_forecaster import TimesFMForecaster

f = TimesFMForecaster(max_context=512)
f.fit(history)

# Point forecast
pt = f.predict(horizon)
print("Point forecast:", pt)

# Quantile forecast — bypass the silent try/except to see raw errors
try:
    result = f.predict_quantiles(horizon)
    if result is None:
        print("predict_quantiles returned None")
    else:
        levels, qvals = result
        print("levels :", levels)
        print("qvals shape:", qvals.shape)
        print("qvals  :", qvals)
except Exception as e:
    import traceback
    print("Exception in predict_quantiles:")
    traceback.print_exc()